# Day 14: Word2Vec — 词向量与分布式语义

用 numpy 从零实现 Skip-gram + Negative Sampling，在 Tiny Shakespeare 数据集上训练词嵌入。

主要概念：
- 分布式语义假说 (Distributional Hypothesis)
- Skip-gram 模型结构
- Negative Sampling 训练方法
- 词向量的 PCA 可视化


In [1]:
import numpy as np
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from models.word2vec import Word2Vec

np.random.seed(42)

In [2]:
# 加载 Tiny Shakespeare
import urllib.request
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
path = '../data/tinyshakespeare.txt'
if not os.path.exists(path):
    urllib.request.urlretrieve(url, path)
with open(path, 'r', encoding='utf-8') as f:
    text = f.read()

text = text[:300000]
print(f'Loaded {len(text)} characters')

Loaded 300000 characters


### 训练 Word2Vec (Skip-gram + Negative Sampling)

In [3]:
w2v = Word2Vec(embedding_dim=64, lr=0.05, epochs=8, window=3,
               n_negative=10, min_count=5, subsample_threshold=1.0)
w2v.fit(text)

[Word2Vec] Vocab: 1184 words (58954 training tokens)


  Epoch 1/8  pairs=353712  loss=3.3543


  Epoch 2/8  pairs=353712  loss=3.1771


  Epoch 3/8  pairs=353712  loss=3.1487


  Epoch 4/8  pairs=353712  loss=3.1275


  Epoch 5/8  pairs=353712  loss=3.1114


  Epoch 6/8  pairs=353712  loss=3.0977


  Epoch 7/8  pairs=353712  loss=3.0862


  Epoch 8/8  pairs=353712  loss=3.0768


### 损失曲线

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, len(w2v.losses_)+1), w2v.losses_, 'o-',
        color='#3498db', lw=1.5, markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Binary Cross-Entropy Loss')
ax.set_title('Word2Vec 训练损失收敛')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 查找相似词

In [5]:
for w in ['king', 'queen', 'man', 'woman', 'night', 'day',
            'love', 'hate', 'life', 'death', 'good', 'evil',
            'sword', 'peace', 'war', 'heart', 'soul', 'god']:
    sim = w2v.most_similar(w, top_n=6)
    words = ', '.join(f'{s[0]}({s[1]:.3f})' for s in sim)
    print(f'{w:10s} → {words}')

king       → richard(0.623), prince(0.536), sovereignty(0.523), ely(0.515), iii(0.503), stanley(0.496)
queen      → margaret(0.686), elizabeth(0.658), words(0.550), wrong(0.530), bitter(0.524), loss(0.520)
man        → little(0.549), sign(0.531), thing(0.518), course(0.505), voice(0.492), greater(0.488)
woman      → humour(0.646), kingly(0.615), scene(0.603), guilty(0.602), presence(0.559), peers(0.559)
night      → comfort(0.524), capitol(0.518), friend(0.514), spoil(0.507), pleasing(0.506), child(0.505)
day        → man's(0.521), many(0.520), next(0.489), lie(0.488), pleasing(0.488), lip(0.459)
love       → heart's(0.606), claim(0.606), stones(0.517), follow'd(0.515), country's(0.503), kiss(0.491)
hate       → hide(0.578), worse(0.527), view(0.525), whether(0.516), turn(0.508), breast(0.493)
life       → wife(0.591), sake(0.588), beauty(0.584), increase(0.569), husband(0.561), labour(0.558)
death      → justice(0.544), nature(0.498), throat(0.497), lost(0.493), charity(0.490), close(

### PCA 可视化

In [ ]:
def pca_2d(X):
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T

# Get embeddings for vocab words
word_list = list(w2v.word_to_idx.keys())
idxs = [w2v.word_to_idx[w] for w in word_list]
embs = w2v.W_in[idxs]
proj = pca_2d(embs)

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(proj[:, 0], proj[:, 1], s=4, alpha=0.4, color='#7f8c8d')

# Label interesting words
labels = ['king', 'queen', 'man', 'woman', 'prince', 'princess',
          'love', 'hate', 'life', 'death', 'night', 'day',
          'good', 'evil', 'sword', 'peace', 'war',
          'god', 'devil', 'heaven', 'hell',
          'father', 'mother', 'brother', 'sister',
          'sun', 'moon', 'star', 'sea', 'fire',
          'eye', 'heart', 'hand', 'soul']

for w in labels:
    if w in w2v.word_to_idx:
        j = word_list.index(w)
        ax.annotate(w, proj[j], fontsize=6, alpha=0.8,
                    xytext=(3, 3), textcoords='offset points')

ax.set_title('Word Embeddings PCA 可视化 (Tiny Shakespeare)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 词向量类比

In [7]:
# 经典类比: king - man + woman ≈ queen
analogies = [('king', 'man', 'queen'),   # 国王 - 男人 + 女人 ≈ 女王
             ('man', 'woman', 'king'),    # 男人 - 女人 + 国王 ≈ ???
             ('day', 'night', 'sun'),     # 白天 - 晚上 + 太阳 ≈ ???
             ('brother', 'sister', 'father')]  # 兄弟 - 姐妹 + 父亲 ≈ 母亲

for a, b, c in analogies:
    res = w2v.analogy(a, b, c, top_n=5)
    out = ', '.join(f'{w}({s:.3f})' for w, s in res)
    print(f'{a} - {b} + {c} → {out}')

king - man + queen → sign(0.501), world(0.465), scarce(0.449), fellow(0.444), note(0.417)
man - woman + king → catesby(0.536), stabb'd(0.499), humour(0.496), force(0.481), plantagenet(0.463)
day - night + sun → lancaster(0.521), sovereignty(0.489), goes(0.480), sudden(0.473), dishonour'd(0.472)
brother - sister + father → bitter(0.462), england's(0.458), spoil(0.445), god's(0.443), edward's(0.439)


### 总结

Word2Vec 的核心思想：
1. **分布式语义假说**：词的语义由其上下文决定
2. **Skip-gram**：用中心词预测上下文
3. **Negative Sampling**：将多分类转化为二分类 + 负采样，大幅提升效率

Word2Vec 的局限：
- 静态词向量（一词一义），无法处理多义词
- 基于窗口的上下文，无法捕捉长距离依赖
- 这些局限将在 Transformer 中得到解决

下一篇文章预告：**Transformer — Attention Is All You Need**